# Iceberg Basics — 06: Schema Evolution


Iceberg has the richest schema evolution support of any table format.
Column operations are metadata-only — no data rewrite required.

Topics: ADD/DROP/RENAME/REORDER columns, type promotion, column IDs vs names.


In [ ]:
import os, time, pathlib, shutil, random, datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/iceberg_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('iceberg-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
spark.sql("CREATE DATABASE IF NOT EXISTS local.icedb")
print(f"Spark {spark.version} | Iceberg catalog ready")

random.seed(42)
N = 100_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2024, 1, 1)
rows = []
for i in range(N):
    qty   = random.randint(1, 10)
    price = round(random.uniform(5, 1000), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 364))
    rows.append((i+1, random.randint(1,10000),
                 f"Product_{random.randint(1,200)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, round(qty*price, 2),
                 random.choice(STATUSES), d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset ready: {N:,} rows")

## Step 1 — Create Table and Add Columns

ADD COLUMN is always metadata-only in Iceberg.

In [ ]:

spark.sql("DROP TABLE IF EXISTS local.icedb.schema_orders")
df.writeTo("local.icedb.schema_orders").using("iceberg").createOrReplace()
print("Original schema:")
spark.table("local.icedb.schema_orders").printSchema()

spark.sql("""
    ALTER TABLE local.icedb.schema_orders
    ADD COLUMNS (
        discount_pct  DOUBLE  COMMENT 'Discount applied 0.0-1.0',
        loyalty_pts   INT     COMMENT 'Loyalty points earned',
        channel       STRING  COMMENT 'Sales channel: web/mobile/store'
    )
""")
print("\nAfter ADD COLUMNS (3 new columns, NO data rewrite):")
spark.table("local.icedb.schema_orders").printSchema()
print()
print("Old rows have null for new columns:")
spark.table("local.icedb.schema_orders").select("order_id","revenue","discount_pct","channel").show(5)


## Step 2 — RENAME and DROP Columns

Rename and drop are also metadata-only.

In [ ]:

spark.sql("ALTER TABLE local.icedb.schema_orders RENAME COLUMN channel TO sales_channel")
print("Renamed: channel → sales_channel")

spark.sql("ALTER TABLE local.icedb.schema_orders DROP COLUMN loyalty_pts")
print("Dropped: loyalty_pts")
spark.table("local.icedb.schema_orders").printSchema()


## Step 3 — Type Promotion

Safe widening promotions: int→long, float→double.

In [ ]:

TYPE_TABLE = "local.icedb.type_test"
spark.sql(f"DROP TABLE IF EXISTS {TYPE_TABLE}")
spark.createDataFrame([(i, float(i), f"name_{i}") for i in range(1000)],
    ["id","value","name"]).writeTo(TYPE_TABLE).using("iceberg").createOrReplace()

spark.sql(f"ALTER TABLE {TYPE_TABLE} ALTER COLUMN id TYPE BIGINT")
print("INT → BIGINT: safe widening ✅")
spark.sql(f"ALTER TABLE {TYPE_TABLE} ALTER COLUMN value TYPE DOUBLE")
print("FLOAT → DOUBLE: safe widening ✅")
spark.table(TYPE_TABLE).printSchema()
print(f"All {spark.table(TYPE_TABLE).count()} rows readable after type change")


## Step 4 — Iceberg Column IDs (vs Names)

Iceberg uses stable numeric IDs, not names, to track columns.

In [ ]:

# Show column IDs from Iceberg metadata
import json as pyjson, pathlib
warehouse = "/workspace/data/warehouse/iceberg"
meta_dir = pathlib.Path(f"{warehouse}/icedb/schema_orders/metadata")
meta_files = sorted(meta_dir.glob("*.json"))
if meta_files:
    meta = pyjson.loads(meta_files[-1].read_text())
    schema = meta.get("current-schema", meta.get("schema", {}))
    print("Iceberg column IDs (stable identifiers):")
    for field in schema.get("fields", []):
        print(f"  id={field['id']:<4} name={field['name']:<20} type={field['type']}")
    print()
    print("When you RENAME a column, the id stays the same.")
    print("Files written before the rename are still readable — no data rewrite.")
    print("This is why Iceberg rename is truly safe — file readers use IDs, not names.")


## Summary



In [ ]:

print("""
ALTER TABLE local.db.table ADD COLUMNS (new_col STRING)
ALTER TABLE local.db.table RENAME COLUMN old TO new
ALTER TABLE local.db.table DROP COLUMN col_name
ALTER TABLE local.db.table ALTER COLUMN id TYPE BIGINT

Safe type promotions: INT→LONG, FLOAT→DOUBLE, DECIMAL(p,s)→DECIMAL(p2,s) if p2>p
Iceberg uses column IDs internally → rename is truly metadata-only
Old data files automatically readable with new schema (IDs match)
""")
